<a href="https://colab.research.google.com/github/suhailabdulwasi/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [ ]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [ ]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [ ]:
before = len(df)

df = df.drop_duplicates()

removed = before - len(df)

log('duplicates', 'dropped duplicate rows', removed)

[duplicates] dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [ ]:
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

log('price', 'cleaned price and converted to float', len(df))

[price] cleaned price and converted to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [ ]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

before = len(df)
df = df[df['qty'].notna() & (df['qty'] >= 0)]
removed = before - len(df)

log('qty', 'dropped missing and negative quantities', removed)

[qty] dropped missing and negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [ ]:
# print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

df['item'] = df['item'].replace(ITEM_MAP)
log('item', 'standardized item names', len(df))


[item] standardized item names (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [ ]:
df['category'] = df['category'].replace({'food':'Food', 'rain-gear':'RainGear', 'Apparel':'Merch'})
log('category', 'standardized categories and treated Apparel as Merch', len(df))

[category] standardized categories and treated Apparel as Merch (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [ ]:
assert df.duplicated().sum() == 0
assert pd.api.types.is_numeric_dtype(df['price'])
assert pd.api.types.is_numeric_dtype(df['qty'])
assert df['qty'].notna().all()
assert (df['qty'] >= 0).all()

assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category'].unique()) == {'Food', 'Merch', 'RainGear'}

### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [ ]:
#
df['revenue'] = df['qty'] * df['price']

revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)

print(revenue_by_category)
print('Overall total:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Overall total: 4740.0


What I would tell the vendor: I would tell the vendor to stock more Merch because it generated the highest revenue at $1,111.50.

### TODO 8 — read back your log

In [ ]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped duplicate rows,15
1,price,cleaned price and converted to float,300
2,qty,dropped missing and negative quantities,25
3,item,standardized item names,275
4,category,standardized categories and treated Apparel as...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

A. Dropping duplicates changed revenue the most, from $4,852.50 to $4,594.50.

B. I chose to combine Apparel with Merch. Another reasonable choice would be to keep Apparel as its own category, which would split the reported revenue between Apparel and Merch. I combined them because I treated apparel as merchandise for reporting purposes.